# Weigh a planet: first one you made, then a real one

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/04_weigh_a_planet.ipynb)

You fit an orbit to radial velocities twice. First to velocities you make yourself, where you know the answer, so you can see what a fit gets right and what it only appears to. Then to a real planet from the TESS-Keck Survey, whose answer is published.

Part 1 fits a planet you make yourself. Part 2 weighs a real planet, checks your answer against the paper, and puts it together with your radius from notebook 3. An optional Extra at the end explains a swing you will see in the worked example.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. At the end of each part, one cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, draws some of the
# plots and records your answers. The science calls are in the cells, where you can read them.
if importlib.util.find_spec("habtrain") is None:
    %pip install -q "habtrain @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import habtrain as ht  # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on
import matplotlib.pyplot as plt  # plots
import numpy as np  # arrays and arithmetic

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/04_weigh_a_planet.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Part 1: Fit a planet you made yourself

Three terms, used from here on.

- **Posterior:** how likely each value of a parameter is, given the data and what you allowed beforehand (here, for example, that K cannot be negative). A fit gives you a shape, not a single number.
- **MCMC** (Markov chain Monte Carlo): a way of mapping that shape. Many walkers wander through the possible values and spend more time where the model fits the data better. This notebook uses emcee ([Foreman-Mackey et al. 2013](https://scixplorer.org/abs/2013PASP..125..306F/abstract)).
- **Corner plot:** every fitted parameter plotted against every other, plus each one on its own. Read it to see which parameters trade off against each other.

The model is a circular orbit: the star's velocity swings as a sine wave with amplitude K, the semi-amplitude from notebook 1, and the period P. The fit also finds an **offset**, the star's average velocity, and a **jitter**, extra scatter beyond the error bars that stands for everything the model leaves out.

The code follows emcee's own tutorial, [Fitting a model to data](https://emcee.readthedocs.io/en/stable/tutorials/line/) (Dan Foreman-Mackey and contributors, MIT licence, kept in `LICENSES/emcee.txt`): the model and its log probability written out, a least-squares answer to start from, then the walkers.

> **Write down: the planet you make**
>
> - What wobble do you give your planet, in m/s? Start with something clear, such as `10`. Put it in `TRUE_K`.
> - What period, in days? Such as `7`. Put it in `TRUE_P`.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
TRUE_K = ...  # the wobble you give your planet, in m/s
TRUE_P = ...  # its period, in days

# The planet the cells below make: yours, or 10 m/s and 7 days if either is left empty.
K_made = ht.record.fill(TRUE_K, 10.0, "TRUE_K")
P_made = ht.record.fill(TRUE_P, 7.0, "TRUE_P")
ht.record.save("made", {"K (m/s)": TRUE_K, "P (days)": TRUE_P}, "the planet you made")

In [ ]:
def circular_orbit(times_days, K, period_days, T0):
    """The star's velocity (m/s) on a circular orbit: zero, and falling, at the transit time T0."""
    return -K * np.sin(2 * np.pi * (times_days - T0) / period_days)


ERROR_MS = 2.0  # each velocity's error bar, as a good spectrograph gives
JITTER_MS = 2.0  # extra scatter from the star, beyond the error bar


def make_velocities(K, period_days, n_nights=40, seed=1):
    """Your planet's velocities on random nights over 1,000 days, as one telescope measures them.

    Returns three arrays: the times (days), the velocities (m/s) and each one's error bar (m/s).
    """
    # the same "random" nights every run, so yours match the answers
    rng = np.random.default_rng(seed)
    times_days = np.sort(rng.uniform(0, 1000, n_nights))
    errors_ms = np.full(n_nights, ERROR_MS)
    scatter_ms = np.hypot(ERROR_MS, JITTER_MS)  # the error bar and the jitter, added in quadrature
    noise_ms = rng.normal(0, scatter_ms, n_nights)
    velocities_ms = circular_orbit(times_days, K, period_days, 0.0) + noise_ms
    return times_days, velocities_ms, errors_ms

In [ ]:
import pandas as pd

# Make 40 nights of velocities for your planet, with a spectrograph's noise added.
times_days, velocities_ms, errors_ms = make_velocities(K_made, P_made)

# Left: the velocities over time. Right: the same points folded on the period you chose.
# The plot reads the survey's columns, with times as BJD, so the table uses them too.
BTJD = ht.constants.BTJD  # 2457000: TESS and this plot count days from BJD 2457000
made_table = pd.DataFrame({
    "BJD": times_days + BTJD,
    "RV": velocities_ms,
    "eRV": errors_ms,
    "Inst": "telescope",
})
fig = ht.plot.velocities(
    made_table, orbit={"P": P_made, "T0": BTJD}, title="Your planet's velocities, 40 nights"
)
plt.show()

In [ ]:
# The log probability: how well one K, offset and
# jitter fit the velocities, plus what the fit allows.
def log_probability(params, times_days, velocities_ms, errors_ms, period_days, T0):
    K, offset, log_jitter = params  # the jitter is fitted as its log, so it stays positive
    if not 0 < K < 2000:  # a wobble cannot be negative; the fit looks below 2,000 m/s
        return -np.inf
    if not -5 <= log_jitter <= 5:  # jitter from e^-5 to e^5, about 0.007 to 150 m/s
        return -np.inf
    model_ms = circular_orbit(times_days, K, period_days, T0) + offset
    variance = errors_ms**2 + np.exp(2 * log_jitter)  # jitter adds in quadrature to each error bar
    residuals = velocities_ms - model_ms
    return -0.5 * np.sum(residuals**2 / variance + np.log(2 * np.pi * variance))

In [ ]:
def least_squares_start(times_days, velocities_ms, period_days, T0):
    """A straight least-squares fit of a sine and an offset: where the walkers start.

    Starting near the answer means the walkers need not search for it.
    """
    phase = 2 * np.pi * (times_days - T0) / period_days
    columns = np.column_stack([np.sin(phase), np.cos(phase), np.ones(len(times_days))])
    coefficients = np.linalg.lstsq(columns, velocities_ms, rcond=None)[0]
    sine_ms = coefficients[0]
    # coefficients[1], the cosine, lets the curve shift a little in time
    offset_ms = coefficients[2]
    # the model is -K sin, so K is the sine's size; at least 0.5 m/s
    K_start = max(abs(sine_ms), 0.5)
    return np.array([K_start, offset_ms, np.log(2.0)])  # the jitter starts at 2 m/s


start = least_squares_start(times_days, velocities_ms, P_made, 0.0)
print(f"Start: K = {start[0]:.2f} m/s, offset {start[1]:.2f} m/s, jitter 2 m/s")

In [ ]:
import emcee

N_WALKERS = 32
N_STEPS = 4000  # if Converged says False below, run again with more, for example 10000

# Each walker starts a tiny step (0.001) from the least-squares answer, in every parameter.
rng = np.random.default_rng(seed=1)
walker_starts = start + 1e-3 * rng.standard_normal((N_WALKERS, 3))

np.random.seed(1)  # emcee's own random steps, the same every run, so your numbers match the answers
sampler = emcee.EnsembleSampler(
    N_WALKERS, 3, log_probability, args=(times_days, velocities_ms, errors_ms, P_made, 0.0)
)
sampler.run_mcmc(walker_starts, N_STEPS, progress=False)
print(f"{N_WALKERS} walkers have each taken {N_STEPS} steps.")

In [ ]:
# Drop the first quarter of the steps as burn-in, while the walkers spread out from their start.
samples = sampler.get_chain(discard=N_STEPS // 4, flat=True)
K_samples = samples[:, 0]

# Converged: the chain ran for 50 autocorrelation times or more. emcee warns when it is shorter.
AUTOCORR_TIMES = 50  # emcee's own test
autocorr_steps = sampler.get_autocorr_time(quiet=True).max()
converged = N_STEPS > AUTOCORR_TIMES * autocorr_steps

# The middle of the posterior for K, and how far its 68% range reaches below and above.
K_low, K, K_high = np.percentile(K_samples, [16, 50, 84])
minus, plus = K - K_low, K_high - K
print(f"Fitted K = {K:.2f} -{minus:.2f} +{plus:.2f} m/s   (you made {K_made} m/s)")
print("Converged:", converged)

In [ ]:
# Every fitted parameter against every other; the blue lines mark the K you made.
fit_full = pd.DataFrame({
    "K (m/s)": K_samples,
    "offset (m/s)": samples[:, 1],
    "jitter (m/s)": np.exp(samples[:, 2]),
})
fig_corner = ht.plot.corner(fit_full, truths={"K (m/s)": K_made})
ht.record.keep_figure("fake-planet-fitted", "corner", fig_corner)
plt.show()

**Converged** means the walkers ran at least 50 times longer than it takes them to forget where they started (the autocorrelation time). If it says False, run the fit again with more steps, for example `N_STEPS = 10000` in the cell that runs emcee.

The two numbers after K say how far the middle 68% of the posterior reaches below and above the middle value. The blue lines on the corner plot mark the value you made. A 68% range leaves out the true value about one time in three, so a value just outside it is normal, not a mistake.

> **Write down: what the fit gave back**
>
> - Is the K you made inside the fitted range? `"yes"` or `"no"`. Put it in `INSIDE`.
> - Report the result the way you would in a paper: K with its range, in one line. Put it in `REPORTED`.
>
> **Where to look:** the printed K and the corner plot.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
INSIDE = ...  # "yes" or "no"
REPORTED = ...  # one line, in quotes, for example "K = 10.8 (+0.5, -0.5) m/s"

ht.record.save(
    "fitted",
    {"K (m/s)": round(K, 2), "minus": round(minus, 2), "plus": round(plus, 2)},
    "the K your fit gave back",
)
ht.record.save("inside", INSIDE, "whether the K you made is inside the fitted range")
ht.record.save("reported", REPORTED, "your K, as you would report it in a paper")

**Predict first**

You are about to keep only 20 of your 40 nights, picked at random, and fit again. What happens to the fitted K, and to its width?

In [ ]:
# Your prediction: what happens to the fitted K, and to its width, with half the nights?
MY_PREDICTION = ...  # one line, in quotes
ht.record.predict("half", MY_PREDICTION)

In [ ]:
# This only runs once you have made your prediction above.
if ht.record.revealed("half"):
    # Keep 20 of the 40 nights, picked at random (the same 20 every run), and fit them the same way.
    rng = np.random.default_rng(seed=2)
    keep = np.sort(rng.choice(len(times_days), size=20, replace=False))
    data_half = (times_days[keep], velocities_ms[keep], errors_ms[keep], P_made, 0.0)
    # ht.fit.sample runs the emcee cells above again: the same walkers, steps and burn-in.
    start_half = least_squares_start(times_days[keep], velocities_ms[keep], P_made, 0.0)
    fit_half = ht.fit.sample(
        log_probability, start_half, ["K", "offset", "log jitter"], args=data_half
    )
    K_half = ht.fit.estimate(fit_half.samples["K"])
    print(f"40 nights: K = {K:.2f} -{minus:.2f} +{plus:.2f} m/s")
    print(f"20 nights: K = {K_half.median:.2f} -{K_half.minus:.2f} +{K_half.plus:.2f} m/s")

Now make the planet ten times lighter, so its wobble is a tenth as big, and fit the full 40 nights again.

**What you need**

The survey in Part 2 confirmed a new planet by its mass only when the fit put the mass at least **3 sigma** away from zero: the middle of the posterior at least three times its uncertainty, where the uncertainty is half the width of its 68% range (the average of the minus and plus numbers). Below that it still reports the mass, as a weak measurement, and gives only an upper limit when there is barely any signal. Its stated goal was 5 sigma, which it reached for 58 of the 126 planets and candidate signals it fitted ([Polanski et al. 2024](https://arxiv.org/abs/2405.14786)).

In [ ]:
# A planet ten times lighter, so its wobble is a tenth as big, on 40 nights.
times_light, velocities_light, errors_light = make_velocities(K_made / 10, P_made)
data_light = (times_light, velocities_light, errors_light, P_made, 0.0)
start_light = least_squares_start(times_light, velocities_light, P_made, 0.0)
fit_light = ht.fit.sample(
    log_probability, start_light, ["K", "offset", "log jitter"], args=data_light
)

# The middle divided by its uncertainty (half the 68% range): 3 or more is the survey's bar.
K_light = ht.fit.estimate(fit_light.samples["K"])
middle_over_uncertainty = K_light.median / ((K_light.minus + K_light.plus) / 2)
print(
    f"Made: K = {K_made / 10} m/s.   "
    f"Fitted: K = {K_light.median:.2f} -{K_light.minus:.2f} +{K_light.plus:.2f} m/s   "
    f"(middle divided by uncertainty: {middle_over_uncertainty:.1f})"
)

Compare the K you made with the one the fit found. When a signal is small next to the scatter, the fitted value wanders, and because K cannot be negative it more often comes out too big than too small.

> **Write down: what changes when the data gets worse**
>
> - What changed when you used half the nights, and did it match your prediction? Put it in `HALF_DATA`.
> - For the planet ten times lighter: would that count as a confirmed mass? Why? Put it in `LIGHTER`.
>
> **Where to look:** the three fits above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
HALF_DATA = ...  # one line, in quotes
LIGHTER = ...  # one line, in quotes

ht.record.save("half_data", HALF_DATA, "what changed with half the nights")
ht.record.save("lighter", LIGHTER, "whether the lighter planet counts as a confirmed mass")

### Hand in Part 1

Run the cell below. It shows all your answers from Part 1 in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `fake-planet-fitted`, and download them.

In [ ]:
# Your Part 1 answers and corner plot, saved as your hand-in.
ht.record.handin(
    "fake-planet-fitted", ["made", "fitted", "inside", "reported", "half_data", "lighter"]
)

### Background reading

Optional. Every fit in these notebooks uses emcee: [Foreman-Mackey et al. 2013, emcee: The MCMC Hammer](https://scixplorer.org/abs/2013PASP..125..306F/abstract). Look for what a walker is, and why emcee runs many of them at once.

## Part 2: Weigh a real planet, then check your answer against the paper

The TESS-Keck Survey spent three years measuring the masses of planets that TESS found by their transits, with two spectrographs: HIRES on the Keck I telescope and the Levy spectrograph on the Automated Planet Finder (APF). Its final analysis released all of it: 9,204 measurements of 86 stars, 9,000 of them with a velocity (the rest carry only an S-value, the activity measure from notebook 2) ([Polanski et al. 2024](https://arxiv.org/abs/2405.14786)).

One of the survey's planets was confirmed by this lab, which also found its outer companion. [Hill et al. 2024](https://arxiv.org/abs/2402.03498) reports TOI-1386 b, a warm sub-Saturn that transits, and TOI-1386 c, a planet at least as heavy as Saturn on a 228-day orbit that does not transit. Same survey and spectrograph, and a fuller version of the fit you are about to run. If you want to see what the finished work looks like, its radial velocity section is a short read before you start.

The model is the one from Part 1, with three changes for real data:
- The period and the transit time come from the TESS transits and stay fixed. The fit finds K.
- Each spectrograph gets its own offset and its own jitter. The two do not measure the same absolute velocity, and a fit that forces one offset on both data sets gets K wrong.
- It has no term for the star's own activity. The survey modelled activity for a few stars, so for those this fit will not match the paper as closely.

This is the model of RadVel's tutorial [K2-24 Fitting & MCMC](https://radvel.readthedocs.io/en/latest/tutorials/K2-24_Fitting+MCMC.html) (Fulton, Petigura, Blunt and contributors, MIT licence, kept in `LICENSES/radvel.txt`), the code behind many of the survey's own fits: the period and transit time held, and an offset (RadVel's `gamma`) and a jitter (`jit`) for each spectrograph. Here it is written out with emcee, as in Part 1.

In [ ]:
# Every velocity the survey released: star, time, velocity, its error, S-value and telescope.
rvs = ht.data.load_rvs()

# The planets you can fit with this model (one planet,
# circular orbit), with what is known before fitting.
planets = ht.data.planets()
print(len(rvs), "velocities;", len(planets), "planets you can weigh with this model")

### Worked example: TOI-1181 b

A hot Jupiter (a giant planet on an orbit so close to its star that it is very hot) with a two-day orbit and a wobble you cannot miss. This works through every step once; then you repeat them on your own planet.

In [ ]:
TKS_CREDIT = "Data: TESS-Keck Survey release, Polanski et al. 2024, VizieR J/ApJS/272/32."
# TOI-1181 b's published period and transit time, and its star's velocities.
orbit = ht.data.orbit("TOI-1181 b")
toi_1181 = rvs[rvs.TKS == orbit["TKS"]]
print(f"{len(toi_1181)} velocities; period {orbit['P']:.4f} days, fixed from the transits")

# Which periods repeat in the velocities? The dashed line marks the transit period.
# ht.fit.periodogram is notebook 2's Lomb-Scargle, here from 0.5 to 50 days.
periods, power = ht.fit.periodogram(toi_1181.BJD, toi_1181.RV, shortest=0.5, longest=50)
fig = ht.plot.periodogram(
    periods,
    power,
    unit="days",
    title="TOI-1181: which periods repeat in the velocities?",
    mark=orbit["P"],
    credit=TKS_CREDIT,
)
plt.show()

In [ ]:
def velocity_data(table):
    """The arrays a fit needs from a velocity table, with each velocity's telescope as a number.

    The telescopes are numbered in alphabetical order, so with both, APF is 0 and HIRES is 1.
    """
    telescopes = sorted(set(table.Inst))
    return {
        "times": table.BJD.to_numpy(float),
        "velocities": table.RV.to_numpy(float),
        "errors": table.eRV.to_numpy(float),
        "telescopes": telescopes,
        "telescope_number": np.array([telescopes.index(name) for name in table.Inst]),
    }

In [ ]:
def log_probability_per_telescope(params, data, period_days, T0):
    """Part 1's log probability, with one offset and one jitter for each telescope."""
    n_telescopes = len(data["telescopes"])
    K = params[0]
    offsets = params[1 : 1 + n_telescopes]
    log_jitters = params[1 + n_telescopes :]
    if not 0 < K < 2000:
        return -np.inf
    if np.any(log_jitters < -5) or np.any(log_jitters > 5):
        return -np.inf
    # offsets[telescope] gives each velocity its own telescope's
    telescope = data["telescope_number"]
    model_ms = circular_orbit(data["times"], K, period_days, T0) + offsets[telescope]
    variance = data["errors"] ** 2 + np.exp(2 * log_jitters[telescope])
    residuals = data["velocities"] - model_ms
    return -0.5 * np.sum(residuals**2 / variance + np.log(2 * np.pi * variance))

In [ ]:
def least_squares_start_per_telescope(data, period_days, T0):
    """Part 1's least-squares start, with one offset column for each telescope."""
    phase = 2 * np.pi * (data["times"] - T0) / period_days
    columns = [np.sin(phase), np.cos(phase)]
    for number in range(len(data["telescopes"])):
        # 1 or 0 per velocity
        is_this_telescope = (data["telescope_number"] == number).astype(float)
        columns.append(is_this_telescope)  # this telescope's offset
    coefficients = np.linalg.lstsq(np.column_stack(columns), data["velocities"], rcond=None)[0]
    K_start = max(abs(coefficients[0]), 0.5)
    offsets = coefficients[2:]
    log_jitters = np.log(np.full(len(data["telescopes"]), 2.0))  # every jitter starts at 2 m/s
    # One array, in the order the log probability unpacks it: K, the offsets, the log jitters.
    return np.concatenate([[K_start], offsets, log_jitters])

In [ ]:
def fit_circular_orbit(table, period_days, T0, steps=4000):
    """Fit K, and an offset and a jitter per telescope, to a velocity table. P and T0 stay fixed."""
    data = velocity_data(table)
    start = least_squares_start_per_telescope(data, period_days, T0)
    names = ["K (m/s)"]
    names += [f"offset {name} (m/s)" for name in data["telescopes"]]
    names += [f"log jitter {name}" for name in data["telescopes"]]
    fit = ht.fit.sample(
        log_probability_per_telescope, start, names, args=(data, period_days, T0), steps=steps
    )
    for name in data["telescopes"]:  # report each jitter in m/s, not its log
        log_jitter = fit.samples.pop(f"log jitter {name}")
        fit.samples[f"jitter {name} (m/s)"] = np.exp(log_jitter)
    return fit


# Fit every velocity: K, plus one offset and one jitter per telescope.
fit_all = fit_circular_orbit(toi_1181, orbit["P"], orbit["T0"])
K_all = ht.fit.estimate(fit_all.samples["K (m/s)"])
jitter_all = ht.fit.estimate(fit_all.samples["jitter HIRES (m/s)"])
print(f"Every velocity: K = {K_all.median:.1f} m/s, jitter {jitter_all.median:.1f} m/s")

In [ ]:
# The velocities over time, and folded on the period with the fitted orbit drawn through them.
offset_all = ht.fit.estimate(fit_all.samples["offset HIRES (m/s)"])
fig = ht.plot.velocities(
    toi_1181,
    orbit=orbit,
    K=K_all.median,
    offsets={"HIRES": offset_all.median},
    title="TOI-1181 b, every velocity",
    credit=TKS_CREDIT,
)
plt.show()

The periodogram's tallest peak sits on the dashed line at the transit period, so the velocities and the transits agree. But look at the folded plot near phase 0, the middle of the transit (phase runs from -0.5 to 0.5 over one orbit): a crowd of points there does not follow the curve.

Those points come from one night. On 8 September 2020, 40 of these 87 velocities were taken one after another, before, during and after a transit, when the planet crossed in front of its star. The star spins, so one half of it is coming towards you and the other half moving away. While the planet covers part of one half, the light from that half is missing, and the star's measured velocity shifts for a few hours. This is the Rossiter-McLaughlin effect; the Extra at the end of this notebook looks at it closely. This model has no way to describe it, so it does the next best thing: it inflates the jitter, the scatter it cannot explain, to about 27 m/s.

The fix is to leave out the hours of the transit and fit again.

In [ ]:
def hours_from_transit(times_bjd, period_days, T0):
    """Hours from the nearest mid-transit, for each time: the velocities folded on the orbit."""
    orbits_since = (times_bjd - T0) / period_days
    phase = (orbits_since + 0.5) % 1 - 0.5  # from -0.5 to 0.5 of an orbit, the transit at 0
    return phase * period_days * 24

In [ ]:
# Which velocities fall inside a transit, plus a margin either side for the uncertain timing?
MARGIN_HOURS = 0.25
transit_hours = ht.data.transit_duration("TOI-1181 b")
hours_from_mid = hours_from_transit(toi_1181.BJD.to_numpy(), orbit["P"], orbit["T0"])
during = np.abs(hours_from_mid) < transit_hours / 2 + MARGIN_HOURS

# Leave those out, and fit again.
outside = toi_1181[~during]
fit_ex = fit_circular_orbit(outside, orbit["P"], orbit["T0"])
K_ex = ht.fit.estimate(fit_ex.samples["K (m/s)"])
jitter_ex = ht.fit.estimate(fit_ex.samples["jitter HIRES (m/s)"])
print(
    f"Transit lasts {transit_hours:.1f} hours; "
    f"{during.sum()} velocities fall inside it and are left out."
)
print(
    f"Every velocity:        K = {K_all.median:.1f} -{K_all.minus:.1f} +{K_all.plus:.1f} m/s, "
    f"jitter {jitter_all.median:.1f} m/s"
)
print(
    f"Transit hours removed: K = {K_ex.median:.1f} -{K_ex.minus:.1f} +{K_ex.plus:.1f} m/s, "
    f"jitter {jitter_ex.median:.1f} m/s.  Converged: {fit_ex.converged}"
)

In [ ]:
offset_ex = ht.fit.estimate(fit_ex.samples["offset HIRES (m/s)"])
fig = ht.plot.velocities(
    outside,
    orbit=orbit,
    K=K_ex.median,
    offsets={"HIRES": offset_ex.median},
    title="TOI-1181 b, transit hours left out",
    credit=TKS_CREDIT,
)
plt.show()

In [ ]:
G, DAY, M_SUN, M_EARTH = ht.constants.G, ht.constants.DAY, ht.constants.M_SUN, ht.constants.M_EARTH


def mass_from_wobble(K, period_days, star_mass_sun):
    """Notebook 1's wobble formula turned round: the planet's mass in Earth masses.

    For a circular orbit seen edge on, and a planet much lighter than its star. SI units inside.
    """
    orbit_factor = (period_days * DAY / (2 * np.pi * G)) ** (1 / 3)
    return K * orbit_factor * (star_mass_sun * M_SUN) ** (2 / 3) / M_EARTH

In [ ]:
# From K to mass, using the star's mass from the NASA Exoplanet Archive.
star_mass_sun = ht.data.star_mass("TOI-1181 b")
mass_ex = mass_from_wobble(K_ex.median, orbit["P"], star_mass_sun)
mass_ex_err = mass_from_wobble((K_ex.minus + K_ex.plus) / 2, orbit["P"], star_mass_sun)
mass_ex_jupiter = mass_ex / ht.constants.EARTH_MASSES_PER_JUPITER_MASS
print(
    f"Star: {star_mass_sun} solar masses.  Planet: {mass_ex:.0f} ± {mass_ex_err:.0f} Earth masses, "
    f"about {mass_ex_jupiter:.1f} Jupiter masses"
)

# Check against the paper: TOI-1181 b's published mass is 377 ± 17 Earth masses (Polanski et al.
# 2024, table 4). Sigma apart: the difference divided by the two uncertainties added in quadrature.
PUBLISHED_MASS, PUBLISHED_MASS_ERR = 377, 17  # Earth masses
sigma_apart = abs(mass_ex - PUBLISHED_MASS) / np.hypot(mass_ex_err, PUBLISHED_MASS_ERR)
print(
    f"Published: {PUBLISHED_MASS} ± {PUBLISHED_MASS_ERR} Earth masses. "
    f"Yours is {sigma_apart:.1f} sigma from it."
)

With the transit hours left out, the jitter falls to about a third, the uncertainty on K shrinks, and the mass lands within 1 sigma of the published one. **Sigma apart** is the difference between two results divided by their combined uncertainty. You fit the same velocities as the paper with nearly the same model, so under 1 sigma is a match. Over 1, check these in order: the fit had not converged; velocities from inside a transit were left in; the two spectrographs were given one offset; or the survey modelled the star's activity, which this fit does not. The first fit was not obviously broken: it converged and its curve went through the points. The high jitter is what showed something was wrong.

The star's mass comes from the NASA Exoplanet Archive. Because the planet transits, its orbit is seen almost edge on, so K gives the mass itself rather than a lower limit.

### Your planet

Weigh the planet you measured in notebook 3, so you can put its radius and mass together at the end. Choose it in the menu under the map. "Rock or gas?" at the end needs your radius from notebook 3; if you skipped it, run notebook 3 for this planet first.

It is normal for a first fit not to converge. If `Converged` says False, run the fit again with `steps=10000`.

In [ ]:
# The planets you can weigh, numbered, and a menu that circles your choice on the map.
# TOI-1181 b, the worked example, is left out.
planets = planets[planets.planet != "TOI-1181 b"].reset_index(drop=True)
fig = ht.plot.planet_map(planets)
plt.show()
choice = ht.choose.planet_menu(planets)

> **Write down: your planet's velocities**
>
> - Choose your planet in the menu above. If the menu does not appear, put its number from the map in `MY_PICK`.
> - Keep only your star's velocities from `rvs`. Put the code in `mine`.
>
> **Where to look:** the map and the menu above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`orbit["TKS"]` is the id of your star in the survey.

</details>

<details><summary>Stuck? Open hint 2</summary>

`rvs[rvs.TKS == orbit["TKS"]]`

</details>

In [ ]:
MY_PICK = ...  # only if the menu did not appear: your planet's number on the map

# Your planet: what you chose in the menu, else the number you typed, else a random one.
planet = ht.choose.resolve(choice, MY_PICK, table=planets, name_column="planet", what="planet")
ht.record.save("my_planet", planet, "your planet")

# Its published period and transit time, which stay fixed in the fit.
orbit = ht.data.orbit(planet)

mine = ...  # your code: only your star's velocities
mine = ht.record.fill(mine, rvs[rvs.TKS == orbit["TKS"]], "mine")
telescopes = ", ".join(sorted(mine.Inst.unique()))
print(f"{len(mine)} velocities from {telescopes}; period {orbit['P']:.4f} days")

In [ ]:
# How many of them were taken during a transit? If
# any were, leave them out as the worked example did.
my_transit_hours = ht.data.transit_duration(planet)
my_hours_from_mid = hours_from_transit(mine.BJD.to_numpy(), orbit["P"], orbit["T0"])
during_mine = np.abs(my_hours_from_mid) < my_transit_hours / 2 + MARGIN_HOURS
print(f"{during_mine.sum()} of them were taken during a transit.")

# Your star's velocities over time, coloured by telescope.
fig = ht.plot.velocities(mine, title=planet, credit=TKS_CREDIT)
plt.show()

**Before you fit: is it the planet or the star?** Run the two periodograms below, one of the velocities and one of the S-values from the same spectra. As notebook 2 showed, a star's activity cycle can look like a planet.

In [ ]:
# Each spectrograph has its own zero point, so take
# each one's median off before looking for periods.
rv_centred = mine.RV - mine.groupby("Inst").RV.transform("median")
s_centred = mine.SHK - mine.groupby("Inst").SHK.transform("median")

# Periodogram of the velocities: does the tallest peak sit on the transit period (dashed)?
periods_rv, power_rv = ht.fit.periodogram(mine.BJD, rv_centred, shortest=0.5, longest=100)
fig = ht.plot.periodogram(
    periods_rv,
    power_rv,
    unit="days",
    title=f"{planet}: velocities",
    mark=orbit["P"],
    credit=TKS_CREDIT,
)
plt.show()

In [ ]:
# Periodogram of the activity (S-values from the same
# spectra): does the star repeat at that period too?
has_s_value = s_centred.notna()  # a few spectra have no S-value
times_s = mine.BJD[has_s_value]
periods_s, power_s = ht.fit.periodogram(times_s, s_centred[has_s_value], shortest=0.5, longest=100)
fig = ht.plot.periodogram(
    periods_s,
    power_s,
    unit="days",
    title=f"{planet}: activity (S-values)",
    mark=orbit["P"],
    credit=TKS_CREDIT,
)
plt.show()

> **Write down: planet or star**
>
> - Does the velocity peak sit on the transit period, and does the activity show the same period? What do you conclude? One or two lines. Put it in `PLANET_OR_STAR`.
>
> **Where to look:** the two periodograms, at the dashed line.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
PLANET_OR_STAR = ...  # one or two lines, in quotes

ht.record.save("planet_or_star", PLANET_OR_STAR, "whether the signal is the planet or the star")

> **Write down: your planet's wobble and mass**
>
> - Fit your star's velocities, with each spectrograph separate. Put the code in `my_fit`. The cell then prints K, whether the fit converged, and your planet's mass, and keeps them for your hand-in.
>
> **Where to look:** the printed K, the corner plot and the folded plot.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

Copy the worked example: `fit_circular_orbit(mine, orbit["P"], orbit["T0"])`.

</details>

<details><summary>Stuck? Open hint 2</summary>

If any of your velocities fall inside a transit, leave them out as the worked example did: use `mine[~during_mine]` in place of `mine`.

</details>

In [ ]:
my_fit = ...  # your code: the fit of your star's velocities, each spectrograph separate

if ht.record.filled(my_fit):
    # K and its 68% range, then the mass from K and the star's mass, with the mass's uncertainty.
    K_mine = ht.fit.estimate(my_fit.samples["K (m/s)"])
    K_mine_err = (K_mine.minus + K_mine.plus) / 2
    star_mass_sun = ht.data.star_mass(planet)
    my_mass = mass_from_wobble(K_mine.median, orbit["P"], star_mass_sun)
    my_mass_err = mass_from_wobble(K_mine_err, orbit["P"], star_mass_sun)
    sigma_from_zero = K_mine.median / K_mine_err
    print(
        f"K = {K_mine.median:.2f} -{K_mine.minus:.2f} +{K_mine.plus:.2f} m/s, "
        f"which is {sigma_from_zero:.1f} sigma from zero.  Converged: {my_fit.converged}"
    )
    print(
        f"Star {star_mass_sun} solar masses.  "
        f"Mass = {my_mass:.1f} ± {my_mass_err:.1f} Earth masses "
        "(the ± is from K only; the star's own mass uncertainty adds more)"
    )
    if sigma_from_zero < 3:
        print(
            "Under 3 sigma: treat this mass as weak, close to an upper limit, "
            "and say so in ROCK_OR_GAS."
        )
else:
    my_mass = ...  # nothing to weigh until you have fitted

In [ ]:
# The corner plot, then the velocities folded with your fitted orbit through them.
if ht.record.filled(my_fit):
    my_corner = ht.plot.corner(my_fit.samples, credit=TKS_CREDIT)
    ht.record.keep_figure("real-planet-fitted", "corner", my_corner)
    plt.show()
    my_offsets = {}
    for name in sorted(set(mine.Inst)):
        my_offsets[name] = ht.fit.estimate(my_fit.samples[f"offset {name} (m/s)"]).median
    my_folded = ht.plot.velocities(
        mine, orbit=orbit, K=K_mine.median, offsets=my_offsets, title=planet, credit=TKS_CREDIT
    )
    ht.record.keep_figure("real-planet-fitted", "folded", my_folded)
    plt.show()

In [ ]:
if ht.record.filled(my_fit):
    my_wobble = {
        "K (m/s)": round(K_mine.median, 2),
        "minus": round(K_mine.minus, 2),
        "plus": round(K_mine.plus, 2),
    }
    ht.record.save("wobble", my_wobble, "your fitted K")
    my_mass_answer = {
        "mass (Earth masses)": round(my_mass, 1),
        "uncertainty": round(my_mass_err, 1),
    }
    ht.record.save("mass", my_mass_answer, "your planet's mass")
    ht.record.save("converged", str(my_fit.converged), "whether your fit converged")

### Rock or gas?

You now have both halves: the radius from the dip in notebook 3, and the mass from the wobble here.

**What you need**

Density alone does not settle it, because rock compresses: the bigger a rocky planet is, the denser it gets, so a big rocky planet can be much denser than Earth. The better test is to ask what radius rock would give a planet of your mass. For a planet made of rock and iron, [Zeng, Sasselov and Jacobsen 2016](https://arxiv.org/abs/1512.08827) give

radius = (1.07 - 0.21 × core fraction) × mass<sup>1/3.7</sup>, in Earth units, where the core fraction is the share of the planet's mass in its iron core,

fitted for about 1 to 8 Earth masses. Earth's core fraction is about a third (0.325). A core fraction of 0, no iron at all, gives the biggest a rocky planet of that mass can be. **If your planet is bigger than that, rock alone cannot make it: it needs something lighter around it, such as gas or water.** Above 8 Earth masses, treat the line as a rough guide. Your radius from notebook 3 usually runs a little high, and density goes as radius cubed, so a density worked out from it runs low.

The lab's 0.8 Earth radii line, from notebook 1, is about rocky planets at Earth's distance from a sun-like star.

> **Write down: what your planet is made of**
>
> - What radius did you measure in notebook 3, in Earth radii? Put it in `MY_RADIUS`. (Use the planet you weighed here.)
> - The cell then plots your planet against what rock could make, and prints the biggest rocky radius for your mass.
> - In one or two lines: is your planet rock, does it need something lighter, or is it too close to the line to say? Could it ever test the lab's line? Put it in `ROCK_OR_GAS`.
>
> **Where to look:** the plot the cell draws, and your mass printed above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
def rocky_radius(mass_earth, core_fraction):
    """The radius (Earth radii) of a planet of rock and iron, for its mass (Earth masses).

    Zeng, Sasselov and Jacobsen 2016, the formula above. A core fraction of 0 is no iron at all.
    """
    return (1.07 - 0.21 * core_fraction) * mass_earth ** (1 / 3.7)


EARTH_CORE_FRACTION = 0.325  # the share of Earth's mass in its iron core

In [ ]:
MY_RADIUS = ...  # your radius from notebook 3, in Earth radii
ROCK_OR_GAS = ...  # one or two lines, in quotes

# This needs your own fit from above (my_mass) and your radius from notebook 3.
if not ht.record.filled(my_mass):
    print("Run your own fit above first: this needs your planet's mass.")
can_compare = ht.record.filled(my_mass) and ht.record.filled(MY_RADIUS)
if can_compare:
    # The biggest a rocky planet of your mass can be (no iron), and an Earth-like mix.
    biggest_rock = float(rocky_radius(my_mass, 0.0))  # no iron at all
    earth_like = float(rocky_radius(my_mass, EARTH_CORE_FRACTION))
    density = my_mass / MY_RADIUS**3 * ht.constants.EARTH_DENSITY_G_CM3  # g/cm3
    print(
        f"Mass {my_mass:.1f} Earth masses, radius {MY_RADIUS} Earth radii, "
        f"density {density:.2f} g/cm3"
    )
    print(
        f"Rock alone would give at most {biggest_rock:.2f} Earth radii at this mass "
        f"(Earth-like mix: {earth_like:.2f})"
    )
ht.record.save(
    "rock_or_gas",
    ROCK_OR_GAS,
    "rock or something lighter, and whether it could test the lab's line",
)

In [ ]:
# Your planet against what rock could make, and your density and the biggest rocky radius saved.
if can_compare:
    fig_rock = ht.plot.mass_radius(my_mass, MY_RADIUS, label=planet, mass_err=my_mass_err)
    ht.record.keep_figure("real-planet-fitted", "rock_or_gas", fig_rock)
    plt.show()
    ht.record.save("density", round(density, 2), "your planet's density (g/cm3)")
    ht.record.save(
        "biggest_rock",
        round(biggest_rock, 2),
        "the biggest rocky radius for your mass (Earth radii)",
    )

### Hand in Part 2

Run the cell below. It shows all your answers from Part 2 in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `real-planet-fitted`, and download them.

In [ ]:
# Your Part 2 answers and plots, saved as your hand-in.
ht.record.handin(
    "real-planet-fitted",
    [
        "my_planet",
        "planet_or_star",
        "wobble",
        "mass",
        "converged",
        "density",
        "biggest_rock",
        "rock_or_gas",
    ],
)

### Background reading

Optional. Your planet's velocities come from [Polanski et al. 2024, The TESS-Keck Survey XX](https://arxiv.org/abs/2405.14786). Figure 3 puts the survey's 126 planets on a mass-radius plot like yours. Look for where your planet sits, and how many reached the survey's goal of a mass measured to 5 sigma.

## Extra: why does the star's velocity jump during a transit?

Optional. This picks up the crowd of TOI-1181 b's velocities near phase 0 that did not follow the orbit in the worked example. The first plot circles them.

**What you need**

A star spins, so one half of it is turning towards you and the other half away. Light from the half coming towards you is shifted slightly blue, light from the other half slightly red, and the two balance out. When a planet crosses in front, it blocks some light from one half first and then from the other. While it blocks part of the half turning towards you, the star looks slightly redshifted and its measured velocity goes up. While it blocks part of the half moving away, the velocity goes down. This is the Rossiter-McLaughlin effect. A planet that orbits in the same direction its star spins crosses the half turning towards you first.

The order of the swing tells you which way the planet goes round compared with the star's spin.

In [ ]:
# This Extra loads its own data, so it runs even if you skipped your own planet in Part 2.
# It uses circular_orbit from Part 1, and hours_from_transit and fit_circular_orbit from Part 2.
rvs = ht.data.load_rvs()
TKS_CREDIT = "Data: TESS-Keck Survey release, Polanski et al. 2024, VizieR J/ApJS/272/32."

# TOI-1181 b again: its orbit, its velocities, and how long each transit lasts.
orbit_1181 = ht.data.orbit("TOI-1181 b")
toi_1181 = rvs[rvs.TKS == orbit_1181["TKS"]]
transit_hours_1181 = ht.data.transit_duration("TOI-1181 b")

# Fit the orbit without the transit hours, as in the worked example.
hours_from_mid = hours_from_transit(toi_1181.BJD.to_numpy(), orbit_1181["P"], orbit_1181["T0"])
during = np.abs(hours_from_mid) < transit_hours_1181 / 2 + MARGIN_HOURS
fit_extra = fit_circular_orbit(toi_1181[~during], orbit_1181["P"], orbit_1181["T0"])
K_extra = ht.fit.estimate(fit_extra.samples["K (m/s)"]).median
offset_extra = ht.fit.estimate(fit_extra.samples["offset HIRES (m/s)"]).median

In [ ]:
from matplotlib.patches import Ellipse

# First, the folded plot from the worked example, with every velocity, and the swing circled.
fig_fold = ht.plot.velocities(
    toi_1181,
    orbit=orbit_1181,
    K=K_extra,
    offsets={"HIRES": offset_extra},
    title="TOI-1181 b, every velocity",
    credit=TKS_CREDIT,
)
ax_fold = fig_fold.axes[1]  # the right-hand, folded panel
# the transit in orbital phase, with room either side
width = transit_hours_1181 / 24 / orbit_1181["P"] * 1.6
ax_fold.add_patch(Ellipse((0, 0), width=width, height=220, fill=False, edgecolor="#d1495b", lw=2))
ax_fold.annotate(
    "this swing",
    (width / 2, 90),
    xytext=(0.12, 125),
    color="#d1495b",
    fontsize=10,
    arrowprops={"arrowstyle": "->", "color": "#d1495b"},
)
plt.show()

In [ ]:
# Now zoom in on that one night: 8 September 2020, when the swing was observed. Julian days start
# at noon, so rounding BJD + 0.5 down numbers each night.
SWING_NIGHT_JD = 2459101  # 8 September 2020
night = toi_1181[np.floor(toi_1181.BJD + 0.5) == SWING_NIGHT_JD]

# Hours from mid-transit, and what is left after taking the orbit out of each velocity.
night_hours = hours_from_transit(night.BJD.to_numpy(), orbit_1181["P"], orbit_1181["T0"])
orbit_ms = circular_orbit(night.BJD, K_extra, orbit_1181["P"], orbit_1181["T0"]) + offset_extra
left_over = night.RV - orbit_ms

In [ ]:
# Plot what is left against hours from mid-transit, with the transit shaded.
fig_swing, ax = plt.subplots(figsize=(9, 3.8), dpi=110)
ax.axvspan(
    -transit_hours_1181 / 2, transit_hours_1181 / 2, color="#f2e8cf", label="during the transit"
)
ax.axhline(0, color="#999", lw=1)
ax.errorbar(night_hours, left_over, night.eRV, fmt=".", color="#3b6ea5")
ax.set_xlabel("Hours from mid-transit")
ax.set_ylabel("Velocity minus the orbit (m/s)")
ax.set_title("TOI-1181 b, the night of 8 September 2020", fontsize=11)
ax.legend(fontsize=8)
fig_swing.text(0.01, 0.005, TKS_CREDIT, fontsize=7, color="#666")
ht.record.keep_figure("transit-swing-read", "swing", fig_swing)
plt.show()

> **Write down: what the swing tells you**
>
> - During the transit, which comes first: an upward swing or a downward one? `"up first"`, `"down first"` or `"unsure"`. Put it in `FIRST_SWING`.
> - So does the planet go round in the same direction the star spins, or the opposite way? `"same"`, `"opposite"` or `"unsure"`, and why in a few words. Put it in `DIRECTION`.
>
> **Where to look:** the shaded part of the plot above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
FIRST_SWING = ...  # "up first", "down first" or "unsure"
DIRECTION = ...  # "same", "opposite" or "unsure", and why

ht.record.save("first_swing", FIRST_SWING, "which swing comes first during the transit")
ht.record.save(
    "direction", DIRECTION, "which way the planet goes round compared with the star's spin"
)

### Hand in the Extra

Run the cell below. It shows your two answers in one table. Hand in the Extra step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `transit-swing-read`, and download them.

In [ ]:
# Your Extra answers and the swing plot, saved as your hand-in.
ht.record.handin("transit-swing-read", ["first_swing", "direction"])